# Satellite tree segmentation (~5m RGB) — SATLAS backbone + fine-tuned head

In [ ]:
import numpy as np, torch
from PIL import Image
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import satlaspretrain_models as spm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SIZE, TARGET_GSD = 256, 5.0

In [ ]:
ds = load_dataset('restor/tcd', cache_dir='./data/tcd')
train_data, test_data = ds['train'], ds['test']

def prep(ex, gsd=TARGET_GSD):
    b, w, h = ex['bounds'], ex['width'], ex['height']
    nw = max(1, int(round((b[2] - b[0]) / gsd)))
    nh = max(1, int(round((b[3] - b[1]) / gsd)))
    img, msk = ex['image'], ex['annotation']
    img = img if isinstance(img, Image.Image) else Image.fromarray(img)
    msk = msk if isinstance(msk, Image.Image) else Image.fromarray(msk)
    img = img.convert('RGB').resize((nw, nh), Image.BILINEAR).resize((SIZE, SIZE), Image.BICUBIC)
    msk = msk.convert('L').resize((nw, nh), Image.NEAREST).resize((SIZE, SIZE), Image.NEAREST)
    x = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
    y = torch.from_numpy((np.array(msk) > 0).astype('int64'))
    return x, y

class TreeDS(Dataset):
    def __init__(self, hf): self.hf = hf
    def __len__(self): return len(self.hf)
    def __getitem__(self, i): return prep(self.hf[i])

train_loader = DataLoader(TreeDS(train_data), batch_size=4, shuffle=True, num_workers=4)
test_loader = DataLoader(TreeDS(test_data), batch_size=4, num_workers=4)

In [ ]:
model = spm.Weights().get_pretrained_model(
    'Sentinel2_Resnet50_SI_RGB', fpn=True, head=spm.Head.SEGMENT, num_categories=2, device=device
).to(device)
for p in model.backbone.parameters():
    p.requires_grad = False
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)

def iou(pred, y):
    p = pred.argmax(1)
    inter = ((p == 1) & (y == 1)).sum().float()
    union = ((p == 1) | (y == 1)).sum().float()
    return ((inter + 1e-6) / (union + 1e-6)).item()

In [ ]:
import os
num_epochs, best = 10, 0.0
save_dir = './local_satlas_tree'; os.makedirs(save_dir, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model.train(); tl = 0.0
    for x, y in tqdm(train_loader, desc=f'{epoch}/{num_epochs} train'):
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item()
    model.eval(); vi = 0.0
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc=f'{epoch}/{num_epochs} val'):
            x, y = x.to(device), y.to(device)
            vi += iou(model(x)[0], y)
    vi /= len(test_loader)
    print(f'epoch {epoch}: train_loss={tl/len(train_loader):.4f} val_iou={vi:.4f}')
    if vi > best:
        best = vi; torch.save(model.state_dict(), f'{save_dir}/satlas_tree.pt')
        print(f'saved (iou={vi:.4f})')

In [ ]:
import matplotlib.pyplot as plt
model.eval()
x, y = prep(test_data[0])
with torch.inference_mode():
    pred = model(x.unsqueeze(0).to(device))[0].argmax(1)[0].cpu().numpy()
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(x.permute(1, 2, 0)); ax[0].set_title('5m image')
ax[1].imshow(y, cmap='gray'); ax[1].set_title('ground truth')
ax[2].imshow(pred, cmap='gray'); ax[2].set_title('prediction')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()